# Trading App V2

This notebook is the live-trading shell for `optimal_trader` going forward.

- `quant-warehouse` owns historical refresh, point-in-time data reads, feature engineering, target engineering, and ThetaData option data.
- `quant-orchestrator` owns feature-family classifier training, option-ranker training, backtests, and strategy artifacts.
- `optimal_trader` owns live broker reconciliation, order planning, Streamlit leaderboard generation, and guarded order submission.

Default behavior is dry-run and artifact reuse. Set the `RUN_*` and `SUBMIT_ORDERS` flags explicitly before doing expensive training or broker submission.

In [ ]:
from __future__ import annotations

from pathlib import Path
import os
import sys

import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "app").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.trading_app_v2_runtime import (
    DEFAULT_STRATEGY_SOURCES,
    alpaca_client_from_env,
    apply_order_gate,
    build_alpaca_equity_orders,
    build_alpaca_option_orders,
    build_latest_equity_leaderboard,
    build_llm_review_orders,
    default_paths,
    load_equity_artifacts,
    run_equity_moe_training,
    run_option_family_ranker_training,
    save_live_artifacts,
    submit_alpaca_orders,
    submit_robinhood_option_orders,
    write_streamlit_leaderboard_app,
)

paths = default_paths(REPO_ROOT)
paths.artifact_root.mkdir(parents=True, exist_ok=True)
paths.live_artifact_dir.mkdir(parents=True, exist_ok=True)
paths

## Environment

This notebook assumes `quant-warehouse`, `quant-orchestrator`, and `tradingagents` are already installed in the `optimal_trader` environment. Dependency installation belongs in environment setup, not in the live trading notebook.


In [ ]:
# Downstream packages are imported by the runtime functions when needed.
# Keep this notebook focused on live-trading parameters and execution.


## Strategy Configuration

This is the 10B universe, all-history training configuration. Oracle trade labels are the side-specific `YE` labels for `k=1..12`.

In [ ]:
RUN_EQUITY_MOE_TRAINING = True
RUN_OPTION_RANKER_TRAINING = False
SUBMIT_ORDERS = False
DRY_RUN = not SUBMIT_ORDERS

DATA_START = "1900-01-01"
DATA_END = ""  # Empty means latest available downstream data.
MIN_MARKET_CAP = 10_000_000_000
TOP_K = 20  # Use 10 or 20 for live candidate breadth.
MIN_LONG_SCORE = 0.50
RUN_BACKTESTS_IN_ORCHESTRATOR = True

ALPACA_EQUITY_ACCOUNT_PREFIX = "EQUITY"
ALPACA_OPTION_ACCOUNT_PREFIX = "OPTION"
ALPACA_LLM_ACCOUNT_PREFIX = "LLM"

OPTION_STRATEGY_ALLOCATION = 100_000.0
OPTION_BUCKET = "otm_option"
OPTION_TENOR_DAYS = 90

# Real Robinhood follows the option strategy with deeply discounted GTC limits.
ROBINHOOD_OPTION_GATE_DISCOUNT_PCT = 90.0

# Equity and LLM real-account gates remain blocked until paper PnL justifies changing them.
ROBINHOOD_EQUITY_GATE_DISCOUNT_PCT = 100.0
ROBINHOOD_LLM_GATE_DISCOUNT_PCT = 100.0

OPTION_PANEL_FOR_RANKER = paths.artifact_root / "option_candidate_panel.parquet"

{
    "artifact_root": str(paths.artifact_root),
    "equity_artifact_dir": str(paths.equity_artifact_dir),
    "option_artifact_dir": str(paths.option_artifact_dir),
    "live_artifact_dir": str(paths.live_artifact_dir),
    "dry_run": DRY_RUN,
}

## Equity MoE Training

Training and backtesting are delegated to `quant-orchestrator`. Data, feature panels, and oracle labels are loaded through `quant-warehouse` inside the orchestrator workflow.

In [ ]:
if RUN_EQUITY_MOE_TRAINING:
    equity_result = run_equity_moe_training(
        artifact_dir=paths.equity_artifact_dir,
        min_market_cap=MIN_MARKET_CAP,
        data_start=DATA_START,
        data_end=DATA_END or None,
        train_end=DATA_END or None,
        top_k_values=(10, 20, 40),
        strategy_sources=DEFAULT_STRATEGY_SOURCES,
        run_backtests=RUN_BACKTESTS_IN_ORCHESTRATOR,
        log_mlflow=False,
    )
    print(equity_result.metrics)

equity_artifacts = load_equity_artifacts(paths.equity_artifact_dir)
strategy_scores = equity_artifacts["strategy_scores"]
display(strategy_scores.head())
display(equity_artifacts["backtest_summary"].head())

## Live Leaderboard

The leaderboard converts orchestrator `strategy_scores.csv` into a live ranking. Latest prices are read from `quant-warehouse`, not from local `optimal_trader` data builders.

In [ ]:
leaderboard = build_latest_equity_leaderboard(
    strategy_scores,
    top_k=TOP_K,
    min_long_score=MIN_LONG_SCORE,
    price_provider="fmp",
)
display(leaderboard.head(TOP_K + 5))

## Option Ranker Training

The option ranker uses a ThetaData option candidate panel plus FMP/FinanceToolkit feature-family panels. The training call goes through the installed `quant_orchestrator` package directly; it does not shell out to downstream repo scripts.

In [ ]:
if RUN_OPTION_RANKER_TRAINING:
    if not OPTION_PANEL_FOR_RANKER.exists():
        raise FileNotFoundError(
            f"Missing option panel {OPTION_PANEL_FOR_RANKER}. Build it with quant-orchestrator option-window workflows first."
        )
    option_ranker_result = run_option_family_ranker_training(
        paths=paths,
        option_panel=OPTION_PANEL_FOR_RANKER,
        output_dir=paths.option_artifact_dir,
        min_market_cap=MIN_MARKET_CAP,
        data_start=DATA_START,
        data_end=DATA_END or None,
        train_end=DATA_END or None,
        target_col="rank_y",
        all_feature_families=True,
    )
    print(option_ranker_result.output_dir)
    display(option_ranker_result.family_summary.head(20))

for path in [
    paths.option_artifact_dir / "selector_summary.csv",
    paths.option_artifact_dir / "basket_summary.csv",
    paths.option_artifact_dir / "family_summary.csv",
]:
    if path.exists():
        print(path)
        display(pd.read_csv(path).head(20))

## Build Broker Order Plans

Three Alpaca paper accounts are managed separately:

- equity paper: equity variant of the feature-family MoE strategy
- option paper: option variant of the same equity signals
- LLM paper: top trades sent to `trading_agents` for review before paper execution

Robinhood real orders are based on the option strategy and use GTC limit orders with the configured downside-protection discount gate.

In [ ]:
paper_order_plans: dict[str, pd.DataFrame] = {}
paper_order_clients: dict[str, object] = {}

try:
    equity_orders = build_alpaca_equity_orders(
        leaderboard=leaderboard,
        account_prefix=ALPACA_EQUITY_ACCOUNT_PREFIX,
        gross_exposure=0.95,
    )
    paper_order_plans["alpaca_equity_paper"] = equity_orders
    paper_order_clients["alpaca_equity_paper"] = alpaca_client_from_env(ALPACA_EQUITY_ACCOUNT_PREFIX)
except Exception as exc:
    print(f"Alpaca equity paper plan skipped: {type(exc).__name__}: {exc}")

try:
    option_plan = build_alpaca_option_orders(
        leaderboard=leaderboard,
        account_prefix=ALPACA_OPTION_ACCOUNT_PREFIX,
        strategy_allocation=OPTION_STRATEGY_ALLOCATION,
        option_bucket=OPTION_BUCKET,
        tenor_days=OPTION_TENOR_DAYS,
    )
    paper_order_plans["alpaca_option_paper"] = option_plan.get("actionable_orders", pd.DataFrame())
    paper_order_clients["alpaca_option_paper"] = option_plan["client"]
except Exception as exc:
    print(f"Alpaca option paper plan skipped: {type(exc).__name__}: {exc}")

try:
    llm_orders = build_llm_review_orders(
        leaderboard=leaderboard,
        top_k=TOP_K,
        account_prefix=ALPACA_LLM_ACCOUNT_PREFIX,
    )
    paper_order_plans["alpaca_llm_paper"] = llm_orders
    paper_order_clients["alpaca_llm_paper"] = alpaca_client_from_env(ALPACA_LLM_ACCOUNT_PREFIX)
except Exception as exc:
    print(f"Alpaca LLM paper plan skipped: {type(exc).__name__}: {exc}")

for name, frame in paper_order_plans.items():
    print(name, len(frame))
    display(frame.head(20))

In [ ]:
robinhood_option_orders = pd.DataFrame()
if "alpaca_option_paper" in paper_order_plans:
    robinhood_option_orders = apply_order_gate(
        paper_order_plans["alpaca_option_paper"],
        gate_discount_pct=ROBINHOOD_OPTION_GATE_DISCOUNT_PCT,
        reference_price_col="limit_price",
    )

display(robinhood_option_orders.head(20))
print({
    "robinhood_option_gate_discount_pct": ROBINHOOD_OPTION_GATE_DISCOUNT_PCT,
    "robinhood_equity_gate_discount_pct": ROBINHOOD_EQUITY_GATE_DISCOUNT_PCT,
    "robinhood_llm_gate_discount_pct": ROBINHOOD_LLM_GATE_DISCOUNT_PCT,
})

## Save Leaderboard And Streamlit App

In [ ]:
saved_paths = save_live_artifacts(
    live_dir=paths.live_artifact_dir,
    leaderboard=leaderboard,
    orders={**paper_order_plans, "robinhood_option_real": robinhood_option_orders},
)
streamlit_app = write_streamlit_leaderboard_app(live_dir=paths.live_artifact_dir)
print(saved_paths)
print(f"Run: streamlit run {streamlit_app}")

## Submit Orders

This cell is inert unless `SUBMIT_ORDERS=True`. Paper Alpaca orders submit to the three configured paper accounts. Robinhood real option orders submit only after the 90 percent discount gate has generated valid GTC limit prices.

In [ ]:
submission_results: dict[str, pd.DataFrame] = {}

for name, orders in paper_order_plans.items():
    client = paper_order_clients.get(name)
    if client is None:
        continue
    submission_results[name] = submit_alpaca_orders(client, orders, dry_run=DRY_RUN)

submission_results["robinhood_option_real"] = submit_robinhood_option_orders(
    robinhood_option_orders,
    dry_run=DRY_RUN,
)

for name, frame in submission_results.items():
    print(name, len(frame))
    display(frame.head(20))